# Humanoid locomotion on GPU — Brax (Colab)

Trains a MuJoCo-style humanoid to walk on a **single GPU** using **Brax** (JAX, GPU-vectorized physics):
~2,048 environments in parallel → **~1h38m on a free/Pro T4** for 100M steps, versus the ~21 h CPU
SAC run in this repo. Measured result: **episode reward ≈ 7,392** (peak ~7,700), and the humanoid
stands and walks.

**This is a different stack from the rest of the repo.** It's Brax's humanoid, not the Gymnasium
`Humanoid-v5` used with Stable-Baselines3 in `rl/`. Same task, different engine — the saved Brax
params are JAX and are **not** loadable by `watch_sac.py`.

### Reproducibility — read before running
Legacy Brax is unmaintained and breaks against current JAX (it calls APIs removed in JAX ≥0.11).
After testing many combinations, the one that works on Colab (Python 3.13) is this **exact pin**:

```
jax[cuda12]==0.4.38   jaxlib==0.4.38   brax==0.12.1
```

Details and every gotcha are in `cloud/README.md`.

**Before running:** Runtime → Change runtime type → **GPU** (a T4 is fine).

In [ ]:
!nvidia-smi

### 1. Install the pinned stack
After this cell finishes, **Runtime → Restart session**, then run the rest from the imports cell.
(The restart is required so the downgraded JAX actually loads.)

In [ ]:
!pip -q install "jax[cuda12]==0.4.38" "jaxlib==0.4.38" "brax==0.12.1"

In [ ]:
# 2. Imports + device check  (run AFTER restarting the session)
import functools
from datetime import datetime
import jax
import matplotlib.pyplot as plt
from brax import envs
from brax.training.agents.ppo import train as ppo
from brax.io import model, html
from IPython.display import HTML, clear_output

print("JAX:", jax.__version__, "| backend:", jax.default_backend())
print("Devices:", jax.devices())   # expect a CudaDevice

In [ ]:
# 3. Train (stability-tuned) ---------------------------------------------
# ctrl_cost_weight is the balance dial. Keep it LOW: a humanoid stays upright by
# making vigorous corrective moves, so over-penalizing control makes it timid and
# fall. 0.1 (default) lets it act to balance -> long episodes, stays up.
env = envs.get_environment(
    "humanoid",
    forward_reward_weight=1.0,      # modest forward drive
    ctrl_cost_weight=0.1,           # low = free to balance (raising this made it fall)
    healthy_reward=5.0,             # strong alive bonus = stay upright longer
    terminate_when_unhealthy=True,
    healthy_z_range=(1.0, 2.0),
)

xdata, ydata = [], []
t0 = datetime.now()
def progress(num_steps, metrics):
    xdata.append(num_steps); ydata.append(metrics["eval/episode_reward"])
    clear_output(wait=True)
    plt.xlabel("env steps"); plt.ylabel("episode reward")
    plt.title(f"reward = {metrics['eval/episode_reward']:.0f}")
    plt.plot(xdata, ydata); plt.show()

train_fn = functools.partial(
    ppo.train,
    num_timesteps=100_000_000, num_evals=15, reward_scaling=0.1,
    episode_length=1000, normalize_observations=True, action_repeat=1,
    unroll_length=10, num_minibatches=32, num_updates_per_batch=8,
    discounting=0.97, learning_rate=3e-4, entropy_cost=1e-3,
    num_envs=2048, batch_size=1024, seed=0,
)
make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)
print("Trained in", datetime.now() - t0)

In [ ]:
# 4. Save the trained params (keep a backup so you never retrain by accident)
model.save_params("humanoid_brax_params", params)
print("saved -> humanoid_brax_params")
try:
    from google.colab import files
    files.download("humanoid_brax_params")
except Exception as e:
    print("download skipped:", e)

### 5. Render + export video — run after a fresh restart

MuJoCo picks its GL backend at the **first import**, so headless rendering needs `MUJOCO_GL=egl`
set *before* any import. The most reliable path on Colab is a clean session:

1. **Runtime → Restart session.**
2. Run the install cell again if needed (packages persist, so usually not).
3. Run the cell below **first** — it sets EGL, skips the GPU for the (tiny) rollout, reloads the
   saved params (no retrain), rolls out, and writes the video.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"       # headless offscreen rendering (set before any import)
os.environ["JAX_PLATFORMS"] = "cpu"   # the rollout is tiny; skip GPU to avoid plugin contention

!pip -q install mediapy
import jax, mediapy as media
from brax import envs
from brax.io import model, image, html
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics
from IPython.display import HTML

# same env settings used in training
env = envs.get_environment(
    "humanoid", forward_reward_weight=1.0, ctrl_cost_weight=0.1,
    healthy_reward=5.0, terminate_when_unhealthy=True, healthy_z_range=(1.0, 2.0),
)

# reload params + rebuild the trained policy (no retrain)
params = model.load_params("humanoid_brax_params")
ppo_net = ppo_networks.make_ppo_networks(
    observation_size=env.observation_size, action_size=env.action_size,
    preprocess_observations_fn=running_statistics.normalize)
inference_fn = ppo_networks.make_inference_fn(ppo_net)(params)

# roll out
jit_reset, jit_step, jit_infer = jax.jit(env.reset), jax.jit(env.step), jax.jit(inference_fn)
rng = jax.random.PRNGKey(0); state = jit_reset(rng); rollout = [state.pipeline_state]
for _ in range(500):
    rng, key = jax.random.split(rng)
    act, _ = jit_infer(state.obs, key)
    state = jit_step(state, act); rollout.append(state.pipeline_state)

# write video (mp4 + webm)
frames = [image.render_array(env.sys, s, width=640, height=480) for s in rollout]
fps = int(round(1.0 / env.dt))
media.write_video("humanoid_brax.mp4",  frames, fps=fps)
media.write_video("humanoid_brax.webm", frames, fps=fps, codec="vp9")
print(f"wrote humanoid_brax.mp4 / .webm  @ {fps} fps, {len(frames)} frames")

try:
    from google.colab import files
    files.download("humanoid_brax.mp4")
except Exception as e:
    print("download skipped (grab it from the Files panel):", e)

# also show it inline
HTML(html.render(env.sys.tree_replace({"opt.timestep": env.dt}), rollout))